<a href="https://colab.research.google.com/github/ArhanTibro/google_colab_works/blob/main/Lunar_Lander_agent_by_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# SWIG: Required to 'wrap' the Box2D C++ physics engine so Python can use it for LunarLander.
# CMake: Manages the build process to compile the physics engine correctly for this machine.
!apt install swig cmake

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 109 not upgraded.


In [26]:
"""
Install:
gymnasium[box2d]: Contains the LunarLander-v2 environment 🌛
stable-baselines3[extra]: The deep reinforcement learning library.
huggingface_sb3: Additional code for Stable-baselines3 to load and upload models from the Hugging Face 🤗 Hub.
"""
!pip install stable-baselines3
!pip install "gymnasium[box2d]"
!pip install pygame>=2.5.2 --only-binary=:all:
!pip install huggingface_sb3

we need to have a virtual screen to be able to render the environment , the following cell will install virtual screen libraries and create and run a virtual screen

In [27]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
#To make sure the new installed libraries are used, sometimes it's required to restart the notebook runtime,
#this cell will force the runtime to crash, so you'll need to connect again and run the code starting from here
import os
os.kill(os.getpid(), 9)

In [28]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
#import the packages
#One additional library we import is huggingface_hub to be able to upload and download trained models from the hub
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [30]:
import gymnasium as gym

# First, we create our environment called LunarLander-v2
env = gym.make("LunarLander-v3") # it is goof to see docs of an environment before implementing

# Then we reset this environment
observation, info = env.reset()

for _ in range(20):
  # Take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # Do this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # Reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

Action taken: 3
Action taken: 2
Action taken: 0
Action taken: 1
Action taken: 1
Action taken: 3
Action taken: 3
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 2
Action taken: 1
Action taken: 2
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 3
Action taken: 0
Action taken: 0


In [31]:
#Let's see what the Environment looks like:

# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("LunarLander-v3")
env.reset()
print("_____OBSERVATION SPACE_____ \n")#here observation mean's state info
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [ 1.7986424  -1.9066231   0.36824852  2.0982356  -3.2454026  -5.671445
  0.65262425  0.36816725]


We see with Observation Space Shape (8,) that the observation is a vector of size 8, where each value contains different information about the lander:

- Horizontal pad coordinate (x),
- Vertical pad coordinate (y),
- Horizontal speed (x),
- Vertical speed (y),
- Angle,
- Angular speed,
- If the left leg contact point has touched the land (boolean)
- If the right leg contact point has touched the land (boolean)

In [32]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 1


The action space (the set of possible actions the agent can take) is discrete with 4 actions available :

- Action 0: Do nothing,
- Action 1: Fire left orientation engine,
- Action 2: Fire the main engine,
- Action 3: Fire right orientation engine.

Reward function (the function that will give a reward at each timestep) :

After every step a reward is granted. The total reward of an episode is the **sum of the rewards for all the steps within that episode**.

For each step, the reward:

- Is increased/decreased the closer/further the lander is to the landing pad.
-  Is increased/decreased the slower/faster the lander is moving.
- Is decreased the more the lander is tilted (angle not horizontal).
- Is increased by 10 points for each leg that is in contact with the ground.
- Is decreased by 0.03 points each frame a side engine is firing.
- Is decreased by 0.3 points each frame the main engine is firing.

The episode receive an **additional reward of -100 or +100 points for crashing or landing safely respectively.**

An episode is **considered a solution if it scores at least 200 points.**

In [33]:
"""
Vectorized Environment
We create a vectorized environment (a method for stacking multiple independent environments into a single environment) of 16 environments, this way, we'll have more diverse experiences during the training.
"""

env = make_vec_env('LunarLander-v3', n_envs=16)

In [34]:
# Define a PPO MlpPolicy architecture
# We use MultiLayerPerceptron (MLPPolicy) because the input is a vector,
# if we had frames as input we would use CnnPolicy

# We added some parameters to accelerate the training
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [14]:
# Now we are training the model for 1000000 times
model.learn( total_timesteps=1000000)

model.name="ppo-LunarLander-v3"
model.save(model.name)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 93.1     |
|    ep_rew_mean     | -151     |
| time/              |          |
|    fps             | 3249     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 97.3        |
|    ep_rew_mean          | -132        |
| time/                   |             |
|    fps                  | 2324        |
|    iterations           | 2           |
|    time_elapsed         | 14          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.008726753 |
|    clip_fraction        | 0.0429      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.37       |
|    explained_variance   | -0.00268    |
|    learning_rate        | 0.

In [35]:
# When you evaluate your agent, you should not use your training environment but create an evaluation environment.

eval_env = Monitor(gym.make("LunarLander-v3"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=-131.96 +/- 40.981418825902324


In [22]:
#login to hugging face account via access token
notebook_login()
!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [39]:
# push the model to hugging face

import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

from huggingface_sb3 import package_to_hub

# PLACE the variables you've just defined two cells above
# Define the name of the environment
env_id = "LunarLander-v3"

# TODO: Define the model architecture we used
model_architecture = "PPO"

## Define a repo_id
## repo_id is the id of the model repository from the Hugging Face Hub
repo_id = "ARHAN2004/lunar_lander_v3_learned_rl_demo"

## Define the commit message
commit_message = "Upload PPO LunarLander-v3 trained agent"

# Create the evaluation env and set the render_mode="rgb_array"
eval_env = DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))])

# PLACE the package_to_hub function you've just filled here
package_to_hub(
    model=model,  # Our trained model
    model_name="ppo-LunarLander-v3",  # The name of our trained model
    model_architecture=model_architecture,  # The model architecture we used: in our case PPO
    env_id=env_id,  # Name of the environment
    # Use the simple make_vec_env instead of manual wrapping
    eval_env = make_vec_env(env_id, n_envs=1),  # Evaluation Environment
    repo_id=repo_id,  # id of the model repository from the Hugging Face Hub (repo_id = {organization}/{repo_name} for instance ThomasSimonini/ppo-LunarLander-v2
    commit_message=commit_message,
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmp5wm_l_d8/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmp5wm_l_d8/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmp5wm_l_d8/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmp5wm_l_d8/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo ARHAN2004/lunar_lander_v3_learned_rl_demo to the Hugging
Face Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r-v3/policy.optimizer.pth: 100%|##########| 1.58kB / 1.58kB            

  ...LunarLander-v3/policy.pth: 100%|##########| 44.1kB / 44.1kB            

  ...-v3/pytorch_variables.pth: 100%|##########| 1.26kB / 1.26kB            

  ...ke/ppo-LunarLander-v3.zip: 100%|##########| 57.2kB / 57.2kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/ARHAN2004/lunar_lander_v3_learned_rl_demo/tree/main/


CommitInfo(commit_url='https://huggingface.co/ARHAN2004/lunar_lander_v3_learned_rl_demo/commit/8200b4ab80e20c8d9a076a483629f2c70aa39996', commit_message='Upload PPO LunarLander-v3 trained agent', commit_description='', oid='8200b4ab80e20c8d9a076a483629f2c70aa39996', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ARHAN2004/lunar_lander_v3_learned_rl_demo', endpoint='https://huggingface.co', repo_type='model', repo_id='ARHAN2004/lunar_lander_v3_learned_rl_demo'), pr_revision=None, pr_num=None)